<a href="https://colab.research.google.com/github/jovanfranco/usdmxn-volatilidad/blob/main/clase15_regresion_multiple_regularizacion.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np
import pandas as pd
import yfinance as yf
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor
from sklearn.linear_model import RidgeCV, LassoCV, LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import TimeSeriesSplit

# 1. Ejemplos a mano
print("Predicción x1=3, x2=4:", round(0.10 + 1.2 * 3 - 0.5 * 4, 2))
print("Residuo con real 1.5:", round(1.5 - (0.10 + 1.2 * 3 - 0.5 * 4), 2))

x = np.array([1, 2, 3.0])
yv = np.array([2, 3, 5.0])
for lam in [0, 1, 7, 14]:
    print("Ridge, lambda", lam, "| pendiente:", round(float((x * yv).sum() / ((x**2).sum() + lam)), 3))
for lam in [0, 14, 30, 46]:
    b = max(0.0, ((x * yv).sum() - lam / 2) / (x**2).sum())
    print("Lasso, lambda", lam, "| pendiente:", round(float(b), 3))

# 2. Sobreajuste con puro ruido: n=20 datos, k=15 variables
rng = np.random.default_rng(42)
n, k = 20, 15
r2, r2aj = [], []
for _ in range(2000):
    m = sm.OLS(rng.normal(size=n), sm.add_constant(rng.normal(size=(n, k)))).fit()
    r2.append(m.rsquared)
    r2aj.append(m.rsquared_adj)
print("R2 promedio con puro ruido:", round(float(np.mean(r2)), 3),
      "| R2 ajustado promedio:", round(float(np.mean(r2aj)), 3))

# 3. Datos reales: USD/MXN contra otros pares
pares = ["MXN=X", "JPY=X", "EURUSD=X", "GBPUSD=X", "CAD=X", "BRL=X"]
datos = yf.download(pares, start="2020-10-07", end="2026-10-22", progress=False)["Close"]
datos["EURUSD=X"] = 1 / datos["EURUSD=X"]
datos["GBPUSD=X"] = 1 / datos["GBPUSD=X"]
rend = datos.pct_change().dropna()
y = rend["MXN=X"]
X = rend.drop(columns="MXN=X")

print("Correlaciones entre las variables explicativas:")
print(X.corr().round(2))
Xc = sm.add_constant(X)
vif = pd.Series([variance_inflation_factor(Xc.values, i) for i in range(1, Xc.shape[1])],
                index=X.columns)
print("VIF:")
print(vif.round(2))

ols = sm.OLS(y, Xc).fit(cov_type="HAC", cov_kwds={"maxlags": 5})
print(ols.summary().tables[1])
print("R2:", round(float(ols.rsquared), 4), "| R2 ajustado:", round(float(ols.rsquared_adj), 4))

# 4. Ridge y lasso, con entrenamiento en el pasado y prueba en el futuro
corte = int(len(X) * 0.7)
Xtr, Xte = X.iloc[:corte], X.iloc[corte:]
ytr, yte = y.iloc[:corte], y.iloc[corte:]
esc = StandardScaler().fit(Xtr)
Xtr_e, Xte_e = esc.transform(Xtr), esc.transform(Xte)

tscv = TimeSeriesSplit(n_splits=5)
modelos = {
    "OLS": LinearRegression(),
    "Ridge": RidgeCV(alphas=np.logspace(-3, 3, 50), cv=tscv),
    "Lasso": LassoCV(cv=tscv, max_iter=100000),
}
for nombre, m in modelos.items():
    m.fit(Xtr_e, ytr)
    print(nombre, "| R2 entrenamiento:", round(float(m.score(Xtr_e, ytr)), 4),
          "| R2 prueba:", round(float(m.score(Xte_e, yte)), 4))

coefs = pd.DataFrame({nombre: m.coef_ for nombre, m in modelos.items()}, index=X.columns)
print("Pendientes con variables estandarizadas:")
print(coefs.round(5))

Predicción x1=3, x2=4: 1.7
Residuo con real 1.5: -0.2
Ridge, lambda 0 | pendiente: 1.643
Ridge, lambda 1 | pendiente: 1.533
Ridge, lambda 7 | pendiente: 1.095
Ridge, lambda 14 | pendiente: 0.821
Lasso, lambda 0 | pendiente: 1.643
Lasso, lambda 14 | pendiente: 1.143
Lasso, lambda 30 | pendiente: 0.571
Lasso, lambda 46 | pendiente: 0.0
R2 promedio con puro ruido: 0.79 | R2 ajustado promedio: 0.005


/tmp/ipykernel_509/2084488106.py:35: FutureWarning: YF.download() has changed argument auto_adjust default to True
  datos = yf.download(pares, start="2020-10-07", end="2026-10-22", progress=False)["Close"]


Correlaciones entre las variables explicativas:
Ticker    BRL=X  CAD=X  EURUSD=X  GBPUSD=X  JPY=X
Ticker                                           
BRL=X      1.00   0.37      0.27      0.30   0.12
CAD=X      0.37   1.00      0.54      0.58   0.28
EURUSD=X   0.27   0.54      1.00      0.76   0.49
GBPUSD=X   0.30   0.58      0.76      1.00   0.45
JPY=X      0.12   0.28      0.49      0.45   1.00
VIF:
Ticker
BRL=X       1.17
CAD=X       1.65
EURUSD=X    2.67
GBPUSD=X    2.71
JPY=X       1.34
dtype: float64
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
const      -6.548e-05      0.000     -0.473      0.636      -0.000       0.000
BRL=X          0.2147      0.022      9.665      0.000       0.171       0.258
CAD=X          0.2777      0.065      4.258      0.000       0.150       0.406
EURUSD=X       0.2470      0.065      3.790      0.000       0.119       0.375
GBPUSD=X       0